In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import time
import os 
import wandb

/Users/mahdikhemakhem/Desktop/github/tweet-impact-predictor/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Simple tweet dataset class
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        # Convert dict of tensors to tensors and remove batch dimension
        input_ids = inputs['input_ids'].squeeze(0)
        attention_mask = inputs['attention_mask'].squeeze(0)
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [5]:
def preprocess_tweet(text):
    """Preprocess tweet by replacing usernames and URLs with placeholders"""
    words = []
    for word in text.split():
        if word.startswith('@'):
            words.append('@user')
        elif word.startswith('http'):
            words.append('http')
        else:
            words.append(word)
    return ' '.join(words)

def analyze_sentiment(text, tokenizer, model, device='cpu'):
    """Analyze sentiment of a single text"""
    # Preprocess text
    text = preprocess_tweet(text)
    
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        prediction = torch.argmax(probs, dim=1).item()
    
    # sentiment_labels = ['Negative', 'Neutral', 'Positive']
    return prediction, probs[0][prediction].item()

def evaluate_model(df, tokenizer, model, device='cpu'):
    """Evaluate model on dataset and calculate average loss"""
    model = model.to(device)
    model.eval()
    correct = 0
    total = 0
    total_loss = 0  # Initialize total loss

    y_true = []
    y_pred = []

    # Create a DataLoader for the evaluation dataset
    eval_dataset = TweetDataset(
        df['tweet'].tolist(),
        df['sentiment'].tolist(),
        tokenizer
    )
    eval_dataloader = DataLoader(eval_dataset, batch_size=16, shuffle=False)

    # Use no_grad for evaluation
    with torch.no_grad():
        for batch in eval_dataloader:
            # Move batch to device
            batch = {k: v.to(device) for k, v in batch.items()}

            # Forward pass
            outputs = model(**batch)
            logits = outputs.logits
            loss = outputs.loss  # Get loss for the batch
            total_loss += loss.item()  # Accumulate loss

            # Predictions
            probs = torch.softmax(logits, dim=1)
            predictions = torch.argmax(probs, dim=1)

            # Collect predictions and labels
            y_true.extend(batch['labels'].cpu().numpy())
            y_pred.extend(predictions.cpu().numpy())

            # Calculate accuracy
            correct += (predictions == batch['labels']).sum().item()
            total += batch['labels'].size(0)

    # Calculate metrics
    accuracy = correct / total
    avg_loss = total_loss / len(eval_dataloader)  # Average loss
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    conf_matrix = confusion_matrix(y_true, y_pred)

    # Print results
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Confusion Matrix:")
    print(conf_matrix)

    return accuracy,precision, recall, f1, avg_loss


test_data_path = "sentiment_test_data.csv"  # Path to test data
test_df = pd.read_csv(test_data_path)
tokenizer = AutoTokenizer.from_pretrained("models/sentiment")
model = AutoModelForSequenceClassification.from_pretrained("models/sentiment")
device = "cpu"

accuracy, precision, recall, f1, avg_loss = evaluate_model(test_df[:40], tokenizer, model, device)

Accuracy: 0.8750
Average Loss: 1.2574
Precision: 0.8765
Recall: 0.8750
F1 Score: 0.8751
Confusion Matrix:
[[12  2  0]
 [ 1 19  1]
 [ 0  1  4]]
